# Social Doors: BIDS events to first-level FEAT

**Author:** Smith Lab  
**Updated:** 2026-08-28  
**License:** MIT

This notebook begins with notebook 01's public fMRIPrep outputs and teaches the repository's core relationship:

```text
BIDS events.tsv
    ↓
FSL 3-column EVs
    ↓
tracked FEAT .fsf template
    ↓
L1 .feat
```

## What you will learn

- How `decision`, `win`, `loss`, and optional `decision-missed` rows become FEAT EVs.
- How a teaching nuisance file differs from the production TEDANA-enhanced file.
- How this repository renders one model-1 activation template for both reward modalities.
- How to inspect the design, FEAT report, and win > loss result.

## Citation and resources

- OpenNeuro: [ds005123 v1.1.3](https://openneuro.org/datasets/ds005123/versions/1.1.3)
- FSL: Jenkinson et al. (2012), *NeuroImage*, 62, 782–790.
- `BIDSto3col.sh`: Tom Nichols and bidsutils contributors.
- Neurodesk example: [Scripted First-Level Analyses in FSL using fMRIPrep data](https://neurodesk.org/edu/examples/functional_imaging/Demo_fmriprep_FEAT.html)

## Table of contents

1. Load software and locate the repository
2. Preview canonical BIDS events
3. Generate FSL 3-column EVs
4. Build simplified teaching confounds
5. Render and run L1 FEAT for both tasks
6. Inspect design, reports, and win > loss
7. Record dependencies

## 1. Load software and locate the repository

In [ ]:
import module

await module.load('fsl/6.0.7.22')
await module.list()

In [ ]:
%pip install -q pandas nibabel matplotlib watermark

from pathlib import Path
import os
import subprocess

import matplotlib.pyplot as plt
import nibabel as nib
import pandas as pd
from IPython.display import IFrame, Image, display

In [ ]:
SUBJECT = '10317'
OUTPUT_SESSION = '01'  # Teaching output namespace; the public BIDS input is sessionless.
TASKS = ('socialdoors', 'doors')
WORKSPACE = Path.home() / 'socialdoors_teaching'
BIDS_DIR = WORKSPACE / 'ds005123'
FMRIPREP_DIR = WORKSPACE / 'derivatives'
FSL_DIR = WORKSPACE / 'fsl'
TEACHING_CONFOUNDS_DIR = WORKSPACE / 'teaching_confounds'

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in search_roots if (p / 'code' / 'L1stats.sh').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Run this notebook from inside a clone of rf1-sra-socdoors.')
for path in (FSL_DIR, TEACHING_CONFOUNDS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO}')
print(f'Teaching workspace: {WORKSPACE}')

## 2. Preview canonical BIDS events

The analysis does not reconstruct behavior from private logs. It consumes the public BIDS `_events.tsv` files directly. `decision` marks the choice period, `win` and `loss` mark feedback, and `decision-missed` is optional because many runs have no missed choice. The public dataset is sessionless even though the production RF1-SRA tree uses `ses-01`; `OUTPUT_SESSION` below is only the teaching output namespace expected by the reusable FEAT worker.

In [ ]:
event_files = {}
for task in TASKS:
    path = (BIDS_DIR / f'sub-{SUBJECT}' / 'func' /
            f'sub-{SUBJECT}_task-{task}_run-1_events.tsv')
    if not path.is_file():
        raise FileNotFoundError(f'Run notebook 01 first; missing {path}')
    event_files[task] = path
    frame = pd.read_csv(path, sep='\t')
    print(f'\n{task}: {path.name}')
    display(frame[['onset', 'duration', 'trial_type']].head(10))
    display(frame['trial_type'].value_counts())

## 3. Generate FSL 3-column EVs

Each output row is `onset  duration  1.0`. The production wrapper assumes sessionful RF1-SRA paths, so this sessionless public example calls the same retained Tom Nichols converter directly. It writes EVs into the standard teaching output layout consumed by `L1stats.sh`; it does not implement a second event-conversion algorithm.

In [ ]:
workflow_env = os.environ.copy()
workflow_env.update({
    'BIDS_ROOT': str(BIDS_DIR),
    'FMRIPREP_ROOT': str(FMRIPREP_DIR),
    'CONFOUNDS_ROOT': str(TEACHING_CONFOUNDS_DIR),
    'FSL_DERIVATIVES_ROOT': str(FSL_DIR),
})
ev_root = FSL_DIR / 'EVfiles' / f'sub-{SUBJECT}' / f'ses-{OUTPUT_SESSION}'
required_evs = ('decision', 'win', 'loss')
for task, events in event_files.items():
    task_dir = ev_root / task
    task_dir.mkdir(parents=True, exist_ok=True)
    outbase = task_dir / 'run-1'
    for old_ev in task_dir.glob('run-1_*.txt'):
        old_ev.unlink()
    subprocess.run([
        'bash', str(REPO / 'code' / 'BIDSto3col.sh'), str(events), str(outbase)
    ], env=workflow_env, check=True)
    missing = [name for name in required_evs if not (task_dir / f'run-1_{name}.txt').is_file()]
    if missing:
        raise RuntimeError(f'EV conversion for {task} omitted required conditions: {missing}')
print(f'Generated EVs under {ev_root}')

In [ ]:
for task in TASKS:
    print(f'\n{task}')
    for ev in sorted((ev_root / task).glob('run-1_*.txt')):
        print(f'  {ev.name}:')
        print(pd.read_csv(ev, sep=r'\s+', header=None, names=['onset', 'duration', 'amplitude']).head())

## 4. Build simplified teaching confounds

> **Production difference:** This nuisance model is simplified for teaching and is not an exact reproduction of the production RF1-SRA analysis, which uses the canonical TEDANA-enhanced confounds from `rf1-sra-linux2`.

For this public exercise we retain cosine regressors, non-steady-state regressors, six rigid-body motion parameters, the first six aCompCor components, and framewise displacement. Missing values are replaced with zero, and the numeric file is written without a header for FEAT.

In [ ]:
def one_match(paths, description):
    paths = list(paths)
    if len(paths) != 1:
        raise RuntimeError(f'Expected one {description}; found {paths}')
    return paths[0]

confound_files = {}
for task in TASKS:
    func = FMRIPREP_DIR / f'sub-{SUBJECT}' / 'func'
    source = one_match(func.glob(f'*task-{task}_run-1*desc-confounds_timeseries.tsv'), f'{task} confounds TSV')
    frame = pd.read_csv(source, sep='\t')
    motion = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
    missing_motion = [column for column in motion if column not in frame]
    if missing_motion:
        raise ValueError(f'Missing required motion columns: {missing_motion}')
    cosine = [column for column in frame if column.startswith('cosine')]
    nonsteady = [column for column in frame if column.startswith('non_steady_state_outlier')]
    acompcor = [column for column in frame if column.startswith('a_comp_cor_')][:6]
    columns = cosine + nonsteady + motion + acompcor + ['framewise_displacement']
    columns = list(dict.fromkeys(column for column in columns if column in frame))
    selected = frame[columns].apply(pd.to_numeric, errors='raise').fillna(0)
    output_dir = TEACHING_CONFOUNDS_DIR / f'sub-{SUBJECT}'
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = f'sub-{SUBJECT}_ses-{OUTPUT_SESSION}_task-{task}_run-1'
    output = output_dir / f'{stem}_desc-TeachingFmriprepConfounds.tsv'
    selected.to_csv(output, sep='\t', index=False, header=False)
    confound_files[task] = output
    print(f'{task}: {selected.shape[0]} time points × {selected.shape[1]} regressors -> {output}')
    print(columns)

## 5. Render and run L1 FEAT for both tasks

The same model structure is used for social and monetary reward. We run activation only—PPI/nPPI is intentionally outside this introductory notebook. The script receives explicit teaching BOLD/confound paths; production defaults remain unchanged.

In [ ]:
def combined_bold(task):
    func = FMRIPREP_DIR / f'sub-{SUBJECT}' / 'func'
    candidates = [
        p for p in func.glob(f'*task-{task}_run-1*space-MNI152NLin6Asym*desc-preproc_bold.nii.gz')
        if 'echo-' not in p.name
    ]
    return one_match(candidates, f'{task} optimally combined MNI BOLD')

bold_files = {task: combined_bold(task) for task in TASKS}
for task, path in bold_files.items():
    print(f'{task}: {path}')

In [ ]:
rendered_files = {}
for task in TASKS:
    command = [
        'bash', str(REPO / 'code' / 'L1stats.sh'), SUBJECT, '1', '0', task,
        '--session', OUTPUT_SESSION, '--bold', str(bold_files[task]),
        '--confounds', str(confound_files[task]), '--render-only'
    ]
    subprocess.run(command, env=workflow_env, check=True)
    rendered = (FSL_DIR / f'sub-{SUBJECT}' / f'ses-{OUTPUT_SESSION}' /
                f'L1_sub-{SUBJECT}_task-{task}_ses-{OUTPUT_SESSION}_model-1_type-act_run-1.fsf')
    rendered_files[task] = rendered
    print(f'\nRendered {task}: {rendered}')
    interesting = [line for line in rendered.read_text().splitlines()
                   if any(token in line for token in ('outputdir', 'npts', 'feat_files(1)', 'confoundev_files(1)', 'custom1', 'custom2', 'custom3', 'custom4', 'conname_real'))]
    print('\n'.join(interesting))

In [ ]:
# Set to True only if you deliberately want to replace an incomplete FEAT directory.
OVERWRITE_INCOMPLETE = False

for task in TASKS:
    command = [
        'bash', str(REPO / 'code' / 'L1stats.sh'), SUBJECT, '1', '0', task,
        '--session', OUTPUT_SESSION, '--bold', str(bold_files[task]),
        '--confounds', str(confound_files[task])
    ]
    if OVERWRITE_INCOMPLETE:
        command.append('--overwrite')
    subprocess.run(command, env=workflow_env, check=True)

## 6. Inspect design, reports, and win > loss

Contrast 4 is the established `win > loss` contrast. The FEAT report is the main QC record; inspect registration, time series, model fit, and residuals rather than relying on a single statistical slice.

In [ ]:
feat_dirs = {}
for task in TASKS:
    feat_dir = (FSL_DIR / f'sub-{SUBJECT}' / f'ses-{OUTPUT_SESSION}' /
                f'L1_task-{task}_ses-{OUTPUT_SESSION}_model-1_type-act_run-1_sm-5.feat')
    feat_dirs[task] = feat_dir
    design = feat_dir / 'design.png'
    report = feat_dir / 'report.html'
    if not design.is_file() or not report.is_file():
        raise FileNotFoundError(f'Incomplete FEAT output: {feat_dir}')
    print(f'\n{task} design')
    display(Image(filename=str(design)))
    display(IFrame(src=str(report), width='100%', height=650))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, task in zip(axes, TASKS):
    zstat = feat_dirs[task] / 'stats' / 'zstat4.nii.gz'
    image = nib.load(zstat)
    data = image.get_fdata()
    z = data.shape[2] // 2
    shown = axis.imshow(data[:, :, z].T, cmap='coolwarm', origin='lower', vmin=-5, vmax=5)
    axis.set_title(f'{task}: win > loss (zstat4), axial z={z}')
    axis.axis('off')
fig.colorbar(shown, ax=axes, shrink=0.7, label='Z')
plt.show()

## 7. Dependencies and next steps

You have now run the two task-specific L1 models used by the production analysis concept. L2 combines these task estimates within participant, but a third/higher-level notebook is intentionally outside this introductory sequence. See `code/README.md` for the production L2/L3 commands and their scientific caveats.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions
await module.list()